In [ ]:
import os
os.chdir('/content')

print("[DeFooocus] Preparing ...")

theme = "dark" #@param ["dark", "light"]
preset = "deafult" #@param ["deafult", "realistic", "anime", "lcm", "sai", "turbo", "lighting", "hypersd", "playground_v2.5", "dpo", "spo", "sd1.5"]
advenced_args = "--share --attention-split --always-cpu --all-in-fp32 --disable-offload-from-vram" #@param {type: "string"}

if preset != "deafult":
  args = f"{advenced_args} --theme {theme} --preset {preset}"
elif preset == "deafult":
  args = f"{advenced_args} --theme {theme}"

# Clean up previous clone completely
!rm -rf /content/DeFooocus
!git clone https://github.com/ehristoforu/DeFooocus.git
%cd /content/DeFooocus

# Upgrade pip first to ensure better wheel matching on Python 3.13
!pip install --upgrade pip

# Safely strip all references of pygit2 to fully bypass pip source compiling
!sed -i '/pygit2/d' requirements_versions.txt

# Modify requirements_versions.txt to remove strict version constraints that trigger source compilation
!sed -i 's/onnxruntime==1.18.1/onnxruntime/g' requirements_versions.txt
!sed -i 's/rembg==2.0.57/rembg/g' requirements_versions.txt
!sed -i '/safetensors/d' requirements_versions.txt
!sed -i '/tokenizers/d' requirements_versions.txt
!sed -i '/transformers/d' requirements_versions.txt
!sed -i '/numpy/d' requirements_versions.txt

# Explicitly install modern compatible versions first via binary wheels
!pip install --no-cache-dir --prefer-binary --upgrade torchsde tokenizers transformers

# Uninstall einops to resolve version conflict before installing requirements
!pip uninstall -y einops

# Install remaining requirements preferring binary wheels
!pip install --prefer-binary --no-cache-dir --upgrade -r requirements_versions.txt

# Patch ui_gradio_extensions.py to fix the Jinja2/Starlette TypeError: unhashable type: 'dict'
# This disables the custom template_response wrapper that causes crash on Python 3.13
with open('modules/ui_gradio_extensions.py', 'r') as f:
    content = f.read()

patched_content = content.replace(
    "return GradioTemplateResponseOriginal(*args, **kwargs)",
    """try:
        return GradioTemplateResponseOriginal(*args, **kwargs)
    except Exception:
        if len(args) > 1 and isinstance(args[1], dict):
            # Starlette/Jinja2 workaround for unhashable dict context in newer versions
            clean_args = list(args)
            clean_args[1] = {k: v for k, v in args[1].items() if k != 'request'}
            try:
                return GradioTemplateResponseOriginal(*clean_args, **kwargs)
            except Exception:
                pass
        return GradioTemplateResponseOriginal(*args, **kwargs)"""
)

with open('modules/ui_gradio_extensions.py', 'w') as f:
    f.write(patched_content)

print("[DeFooocus] Starting ...")
# Bypass the updater launcher to avoid triggering the internal legacy pip install loop
!python launch.py $args

[DeFooocus] Preparing ...
Cloning into 'DeFooocus'...
remote: Enumerating objects: 7032, done.
remote: Total 7032 (delta 0), reused 0 (delta 0), pack-reused 7032 (from 1)
Receiving objects: 100% (7032/7032), 47.35 MiB | 34.61 MiB/s, done.
Resolving deltas: 100% (3825/3825), done.
/content/DeFooocus
Found existing installation: einops 0.8.0
Uninstalling einops-0.8.0:
  Successfully uninstalled einops-0.8.0
INFO: pip is looking at multiple versions of rembg to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of rembg to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
[DeFooocus] Starting ...
[System ARGV] ['launch.py', '--sha